# 🚀 FEDHEP MIDTERM OVERNIGHT EXPERIMENT SUITE ON CIFAR-100

This notebook implements the complete Midterm Development Plan on **CIFAR-100** using **ResNet-9**:

1. **IDEA 1**: Defense Scope Evaluation (`global` vs `cluster` vs `both` vs `none`) under Targeted Attack Tiers (`root_only`, `parent_only`, `all`).
2. **IDEA 2**: Stealthy Byzantine ALIE (*A Little Is Enough*) Attack Suite across **FedAvg**, **Ditto**, **Multi-Krum**, and **Defended FedHEP** ($q \in [0.1, 0.4]$).
3. **IDEA 3**: Peer Cluster Count ($K \in \{1, 2, 3, 5, 8\}$) Sensitivity Sweep across Dirichlet skews ($\alpha \in \{0.5, 0.05\}$).

> **Crash & Timeout Resilience**: Every single run is written and flushed immediately to a dedicated CSV file. If the session times out or disconnects overnight, rerun the cell and it will **automatically skip already-completed runs and pick up right where it left off**!

In [ ]:
# ==============================================================================
# Cell 1: Environment Setup, Auto-Clone / Pull, Accelerator Check & CIFAR-100 Auto-Detection
# ==============================================================================
%load_ext autoreload
%autoreload 2

import os
import sys
import glob
import shutil
import subprocess

# 1. Auto-clone or pull the latest code from GitHub on Kaggle
REPO_URL = "https://github.com/nam200718/Topology-aware-FDL.git"
REPO_NAME = "Topology-aware-FDL"

# If neither 'oreh' nor 'src' is in current directory, find or clone repo
if not os.path.exists("oreh") or not os.path.exists("src"):
    if os.path.exists(REPO_NAME):
        print(f"Found {REPO_NAME}, entering and syncing latest main...")
        os.chdir(REPO_NAME)
        subprocess.run(["git", "fetch", "origin", "main"], check=False)
        subprocess.run(["git", "reset", "--hard", "origin/main"], check=False)
    else:
        print(f"Cloning latest codebase from {REPO_URL}...")
        subprocess.run(["git", "clone", REPO_URL], check=True)
        if os.path.exists(REPO_NAME):
            os.chdir(REPO_NAME)
elif os.path.exists(".git"):
    print("Syncing existing repository with git fetch and reset --hard origin/main...")
    subprocess.run(["git", "fetch", "origin", "main"], check=False)
    subprocess.run(["git", "reset", "--hard", "origin/main"], check=False)

# Ensure repo root and submodules are in sys.path
cwd = os.getcwd()
for p in [cwd, os.path.join(cwd, ".."), ".", "..", "/kaggle/working", "/kaggle/working/Topology-aware-FDL"]:
    abs_p = os.path.abspath(p)
    if os.path.exists(abs_p) and abs_p not in sys.path:
        sys.path.insert(0, abs_p)

# Reload oreh modules if already imported in kernel
if "oreh" in sys.modules:
    import importlib
    import oreh
    importlib.reload(oreh)
    if "oreh.comparison_runner" in sys.modules:
        import oreh.comparison_runner
        importlib.reload(oreh.comparison_runner)
    if "oreh.engine" in sys.modules:
        import oreh.engine
        importlib.reload(oreh.engine)

print(f"Active Working Directory: {os.getcwd()}")

import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"PyTorch Version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# 2. Auto-detect or link pre-added CIFAR-100 dataset on Kaggle
data_dir = "./data"
os.makedirs(data_dir, exist_ok=True)

# Try finding CIFAR-100 in Kaggle input
train_candidates = glob.glob("/kaggle/input/**/cifar-100-python/train", recursive=True) + glob.glob("/kaggle/input/**/train", recursive=True)
target_path = os.path.join(data_dir, "cifar-100-python")

if train_candidates and not os.path.exists(target_path):
    cifar_src_dir = os.path.dirname(train_candidates[0])
    try:
        os.symlink(cifar_src_dir, target_path)
        print(f"Symlinked Kaggle dataset {cifar_src_dir} -> {target_path}")
    except Exception:
        shutil.copytree(cifar_src_dir, target_path)
        print(f"Copied Kaggle dataset {cifar_src_dir} -> {target_path}")

print(f"Dataset directory configured at: {data_dir}")

import oreh
from oreh.comparison_runner import OrehComparisonRunner
print(f"Oreh suite loaded successfully! (has run_single_experiment: {hasattr(OrehComparisonRunner, 'run_single_experiment')})")


In [ ]:
# ==============================================================================
# Cell 2: Pre-Flight Sanity Check (1 Round Test on CIFAR-100)
# ==============================================================================
# Tests end-to-end data loading, ResNet-9 backbone, and ALIE perturbation
# before starting the full overnight run (~15-20 seconds).
from oreh.comparison_runner import OrehComparisonRunner

print("Running 1-round pre-flight sanity check on CIFAR-100...")
sanity_runner = OrehComparisonRunner(
    dataset="cifar100",
    num_rounds=1,
    num_clients=15,
    num_clusters=3,
    alpha=0.5,
    device=device,
    train_subset=1000,
    test_subset=500,
    data_dir=data_dir,
    output_dir="./outputs/oreh_sanity",
)

test_res = sanity_runner.run_single_experiment(
    defense_scope="both",
    attack_target_tier="all",
    attack_type="alie",
    byzantine_rate=0.2,
    seed=42,
)

print("\n Pre-flight sanity check PASSED!")
print(f"Sanity Test Acc: {test_res['final_test_acc']}% | Latency: {test_res['total_defense_latency_ms']} ms | Byz Trust: {test_res['byzantine_trust_score']}")


## 🧪 IDEA 1: Defense Scope Evaluation under Targeted Attacks
Evaluates whether `defense_scope = 'both'` always outperforms `only root` or `only cluster`, particularly when the attacker attacks the Parent head directly.

In [ ]:
# ==============================================================================
# Cell 3: [IDEA 1] Defense Scope vs Targeted Attack Tiers Matrix
# ==============================================================================
# Output: outputs/oreh_eval/cifar100_idea1_scope.csv
from oreh.comparison_runner import OrehComparisonRunner

runner_idea1 = OrehComparisonRunner(
    dataset="cifar100",
    num_rounds=25,
    num_clients=15,
    num_clusters=3,
    alpha=0.5,
    device=device,
    train_subset=10000,
    test_subset=3000,
    data_dir=data_dir,
    output_dir="./outputs/oreh_eval",
)

df_scope = runner_idea1.run_idea1_scope_matrix(
    attack_types=["alie", "sign_flip"],
    scopes=["global", "cluster", "both", "none"],
    target_tiers=["root_only", "parent_only", "all"],
    byzantine_rate=0.20,
    seed=42,
)

print("\n--- IDEA 1 SUMMARY TABLE ---")
cols_s = ["attack_type", "attack_target_tier", "defense_scope", "final_test_acc", "bottom_10_fairness", "total_defense_latency_ms", "byzantine_trust_score"]
print(df_scope[[c for c in cols_s if c in df_scope.columns]].to_string(index=False))


## 🛡️ IDEA 2: ALIE Attack Suite across Baselines (Incorporating Ditto)
Evaluates how stealthy small perturbations (ALIE, Baruch et al., 2019) fool threshold-based defenses across **FedAvg**, **Ditto**, **Multi-Krum**, and **Defended FedHEP**.

In [ ]:
# ==============================================================================
# Cell 4: [IDEA 2] ALIE Attack Suite (FedAvg, Ditto, Multi-Krum, Defended FedHEP)
# ==============================================================================
# Output: outputs/oreh_eval/cifar100_idea2_alie.csv
from oreh.comparison_runner import OrehComparisonRunner

runner_idea2 = OrehComparisonRunner(
    dataset="cifar100",
    num_rounds=25,
    num_clients=15,
    num_clusters=3,
    alpha=0.5,
    device=device,
    train_subset=10000,
    test_subset=3000,
    data_dir=data_dir,
    output_dir="./outputs/oreh_eval",
)

df_alie = runner_idea2.run_idea2_alie_suite(
    methods=["fedavg", "ditto", "multikrum", "fedhep"],
    byzantine_rates=[0.1, 0.2, 0.3, 0.4],
    seed=42,
)

print("\n--- IDEA 2 (ALIE ATTACK SUITE) SUMMARY TABLE ---")
cols_a = ["method", "byzantine_rate", "final_test_acc", "bottom_10_fairness", "byzantine_trust_score", "total_runtime_s"]
print(df_alie[[c for c in cols_a if c in df_alie.columns]].to_string(index=False))


## 🌐 IDEA 3: Number of Peer Clusters ($K$) Sensitivity Sweep
Evaluates whether the number of peer clusters $K$ monotonically improves accuracy, or exhibits an optimal Inverted-U sweet spot across Extreme ($\alpha=0.05$) and Moderate ($\alpha=0.5$) skews.

In [ ]:
# ==============================================================================
# Cell 5: [IDEA 3] Number of Peer Clusters (K) Sensitivity Sweep
# ==============================================================================
# Output: outputs/oreh_eval/cifar100_idea3_k_clusters.csv
from oreh.comparison_runner import OrehComparisonRunner

runner_idea3 = OrehComparisonRunner(
    dataset="cifar100",
    num_rounds=25,
    num_clients=15,
    device=device,
    train_subset=10000,
    test_subset=3000,
    data_dir=data_dir,
    output_dir="./outputs/oreh_eval",
)

df_k = runner_idea3.run_idea3_cluster_sweep(
    k_values=[1, 2, 3, 5, 8],
    alphas=[0.5, 0.05],
    byzantine_rate=0.0,
    seed=42,
)

print("\n--- IDEA 3 (CLUSTER COUNT K) SUMMARY TABLE ---")
cols_k = ["num_clusters", "alpha", "final_test_acc", "bottom_10_fairness", "total_runtime_s"]
print(df_k[[c for c in cols_k if c in df_k.columns]].to_string(index=False))


## 📊 Final Consolidated Report & Visualizations
Loads the three independent CSV result files and generates publication-grade figures.

In [ ]:
# ==============================================================================
# Cell 6: Consolidated Midterm Report & Visualizations
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

output_dir = './outputs/oreh_eval'
fig_dir = './outputs/oreh_figures'
os.makedirs(fig_dir, exist_ok=True)

# 1. Plot Idea 1: Scope vs Attack Target
scope_file = os.path.join(output_dir, 'cifar100_idea1_scope.csv')
if os.path.exists(scope_file):
    df_s = pd.read_csv(scope_file)
    if not df_s.empty and 'attack_type' in df_s.columns:
        df_alie = df_s[df_s['attack_type'] == 'alie']
        if not df_alie.empty:
            plt.figure(figsize=(10, 5))
            scopes_list = df_alie['defense_scope'].unique()
            targets_list = df_alie['attack_target_tier'].unique()
            x = np.arange(len(targets_list))
            width = 0.2
            for i, sc in enumerate(scopes_list):
                sub = df_alie[df_alie['defense_scope'] == sc]
                acc_map = dict(zip(sub['attack_target_tier'], sub['final_test_acc']))
                vals = [acc_map.get(t, 0.0) for t in targets_list]
                plt.bar(x + i * width, vals, width, label=f'Scope: {sc}')
            plt.title('CIFAR-100: Defense Scope Resilience under ALIE Attack Target Tiers')
            plt.ylabel('Personalized Test Accuracy (%)')
            plt.xlabel('Attacker Target Tier')
            plt.xticks(x + width * (len(scopes_list) - 1) / 2, targets_list)
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.legend()
            plt.tight_layout()
            plt.savefig(os.path.join(fig_dir, 'idea1_scope_vs_target.png'), dpi=200)
            plt.show()

# 2. Plot Idea 2: ALIE Attack Suite vs Existing Paper Baselines
alie_file = os.path.join(output_dir, 'cifar100_idea2_alie.csv')
if os.path.exists(alie_file):
    df_a = pd.read_csv(alie_file)
    if not df_a.empty and 'method' in df_a.columns:
        plt.figure(figsize=(9, 5))
        clean_map = {
            'FEDAVG': 40.20, 'DITTO': 43.96, 'MULTIKRUM': 26.77,
            'Defended FedHEP': 47.88, 'FEDHEP': 47.88, 'DEFENDED FEDHEP': 47.88
        }
        for m in df_a['method'].unique():
            sub = df_a[df_a['method'] == m].sort_values('byzantine_rate')
            clean_acc = clean_map.get(str(m).upper(), clean_map.get(m, 40.0))
            rates = [0.0] + list(sub['byzantine_rate'])
            accs = [clean_acc] + list(sub['final_test_acc'])
            plt.plot(rates, accs, marker='o', linewidth=2, label=m)
        plt.title('CIFAR-100: ALIE Attack Robustness Curve (Incorporating Ditto)')
        plt.xlabel('Byzantine Attacker Fraction (q)')
        plt.ylabel('Test Accuracy (%)')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, 'idea2_alie_robustness_curve.png'), dpi=200)
        plt.show()

# 3. Plot Idea 3: Inverted-U Curve of K Clusters
k_file = os.path.join(output_dir, 'cifar100_idea3_k_clusters.csv')
if os.path.exists(k_file):
    df_k = pd.read_csv(k_file)
    if not df_k.empty and 'alpha' in df_k.columns:
        plt.figure(figsize=(8, 5))
        for a in df_k['alpha'].unique():
            sub_k = df_k[df_k['alpha'] == a].sort_values('num_clusters')
            label = f'Extreme Non-IID (alpha={a})' if a <= 0.1 else f'Moderate Non-IID (alpha={a})'
            plt.plot(sub_k['num_clusters'], sub_k['final_test_acc'], marker='s', linewidth=2, label=label)
        plt.title('CIFAR-100: Sensitivity of Peer Cluster Count (K) under Non-IID Skew')
        plt.xlabel('Number of Peer Clusters (K)')
        plt.ylabel('Personalized Test Accuracy (%)')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, 'idea3_cluster_count_curve.png'), dpi=200)
        plt.show()

print('All charts successfully generated and saved to:', fig_dir)
